In [24]:
import phoenix as px 
from phoenix.otel import register 
from opentelemetry import trace 
import uuid
from qdrant_client import QdrantClient 
from qdrant_client.http import models
import google.generativeai as genai

class ObservabilityManage:
    def __init__(self,project_name='rag-advanced'):
        self.session=px.launch_app()
        register(project_name=project_name)
        self.tracer=trace.get_tracer(__name__)
        print('Phonex UI',self.session.url)

    def get_tracer(self):
        return self.tracer 

/var/folders/pw/f9wk46495c1cdw06ky1g2z100000gp/T/ipykernel_1531/3200073714.py:7: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [25]:
# !pip install google-generativeai

**Creating saperate decorators** 

In [27]:
#pdf_reader decorator
def trace_pdf_read(func):
    def wrapper(self,pdf_path):
        with self.tracer.start_as_current_span('pdf_read') as span:
            result=func(self,pdf_path)
            span.set_attribute('pdf_path',pdf_path)
            span.set_attribute('num_pages',len(result))

            return result 
    return wrapper 

In [28]:
#chunking decorator
def trace_chunking(func):
    def wrapper(self,docs):
        with self.tracer.start_as_current_span('chunking') as span:
            chunks=func(self,docs)
            span.set_attribute('num_chunks',len(chunks))
            return chunks 
    return wrapper

In [29]:
#embedding decorator
def trace_embed(funs):
    def wrapper(self,chunks):
        with self.tracer.start_as_current_span('embedding') as span:
            embeddings=funs(chunks)
            span.set_attribute('stored_chunks',len(chunks))
            return embeddings
    return wrapper 


In [30]:
#retrieval decoder 
def trace_retrieval(funs):
    def wrapper(self,query,top_k):
        with self.tracer.start_as_current_span('retrieval') as span:
            retrival=funs(query,top_k)
            span.set_attribute('query',query)
            span.set_attribute('top_k',top_k)
            span.set_attribute('retrieved_count',len(retrival))
            return retrival 
    return wrapper

In [31]:
#re-ranking decorator 
def trace_rerank(func):
    def wrapper(self,query,docs):
        with self.tracer.start_as_current_span('reranking') as span:
            result=func(query,docs)
            span.set_attribute('inputdocs',len(docs))
            span.set_attribute('outputdocs',len(result))

            return result 
    return wrapper 


In [32]:
def trace_llm(func):
    def wrapper(self,prompt):
        with self.tracer.start_as_current_span('llm+generation') as span:
            result=func(prompt)
            span.set_attribute('prompt_length',len(prompt))
            span.set_attribute('respone_length',len(result))
            return result 
        
    return wrapper 

In [33]:
def trace_eval(func):
    def wrapper(self, query, context, answer):
        with self.tracer.start_as_current_span("evaluation") as span:
            score = func(self, query, context, answer)
            span.set_attribute("query", query)
            span.set_attribute("score", score)
            return score
    return wrapper

In [34]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [35]:
class PDFPRocessor:
    def __init__(self,tracer):
        self.tracer=tracer 

    @trace_pdf_read
    def read_pdf(self,pdf_path):
        loader=PyMuPDFLoader(pdf_path)
        text=loader.load() 

        return text 
    @trace_chunking
    def chunk_doc(self,docs):
        splitter=RecursiveCharacterTextSplitter(chunk_size=1000,overlap_size=200)
        all_chunks=[]
        for doc in docs:
            all_chunks.extend(splitter.split_text(doc.page_content))
        return all_chunks

In [36]:
class VectorStore:
    def __init__(self,qdrant,embedder,tracer):
        self.qdrant=qdrant 
        self.embedder=embedder
        self.tracer=tracer 

    @trace_embed
    def store(self,chunks):
        points=[]
        for chunk in chunks:
            vector=self.embedder.encode(chunk).tolist() 

            points.append(
                models.PointStruct(
                    id=str(uuid.uuid4()),
                    vector=vector,
                    payload={'text',chunk}
                )
            )
            self.qdrant.upsert(collection_name='askan',points=points)


In [37]:
class Retriver:
    def __init__(self,embedder,qdrant,tracer):
        self.embedder=embedder
        self.qdrant=qdrant
        self.tracer=tracer 

    @trace_retrieval
    def retrieve(self,query,top_k=10):
        query_vec=self.embedder.encode(query).tolist()
        result=self.qdrant.search(
            collection_name='askan',
            query_vector=query_vec,
            limit=top_k 
        )
        return [r.payload['text'] for r in result]


In [38]:
class Reranker:
    def __init__(self,co,tracer):
        self.co=co 
        self.tracer=tracer 

    @trace_rerank 
    def rerank(self,query,docs):
        result=self.co.rerank(
            query=query,
            documents=docs,
            top_n=3,
            model="rerank-english-v3.0"
        )
        return [docs[r.index] for r in result.results]

In [39]:
class LLM:
    def __init__(self,model,tracer):
        self.model=model 
        self.tracer=tracer 

    @trace_llm
    def generate(self,prompt):
        resp=self.model.generate_content(prompt)
        return resp.text

In [ ]:
class Evaluator:
    def __init__(self,llm,tracer):
        self.llm=llm 
        self.tracer=tracer 

    @trace_eval 
    def evaluate(self,query,context,answer):
        prompt=f"""
Question:{query}
Context: {context}
Answer: {answer}
Is answer correct ? rate 1 or 0
"""
        result=self.llm.generate(prompt)

        return 1 if '1' in result else 0 

In [41]:
class RAGPipeline:
    def __init__(self, retriever, reranker, llm, evaluator):
        self.retriever = retriever
        self.reranker = reranker
        self.llm = llm
        self.evaluator = evaluator

    def run(self, query):
        docs = self.retriever.retrieve(query, top_k=10)
        reranked = self.reranker.rerank(query, docs)

        context = "\n".join(reranked)

        prompt = f"""
        Context:
        {context}

        Question:
        {query}
        """

        answer = self.llm.generate(prompt)

        score = self.evaluator.evaluate(query, context, answer)

        return {
            "answer": answer,
            "score": score
        }

In [42]:
import streamlit as st
import os
from dotenv import load_dotenv

load_dotenv()

True

: 

In [45]:
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer
import cohere
import groq

In [46]:
# Qdrant
qdrant = QdrantClient(
    url=os.getenv("Url"),
    api_key=os.getenv("QDRANT_API_KEY")
)
if not qdrant.collection_exists("askan"):
    qdrant.create_collection(
        collection_name="askan",
        vectors_config=models.VectorParams(
            size=384,
            distance=models.Distance.COSINE
        )
    )

In [47]:
class Embedder:
    def __init__(self):
        self.model = SentenceTransformer("all-MiniLM-L6-v2")

    def encode(self, text):
        return self.model.encode(text)

embedder = Embedder()

/Users/asangtriratnaingle/Desktop/Codes_/embeddings/venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
I0000 00:00:1774075739.407245   12193 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


In [48]:
obs = ObservabilityManage()
tracer = obs.get_tracer()

Existing running Phoenix instance detected! Shutting it down and starting a new instance...
Overriding of current TracerProvider is not allowed


🌍 To view the Phoenix app in your browser, visit http://localhost:6006/
📖 For more information on how to use Phoenix, check out https://arize.com/docs/phoenix
🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: rag-advanced
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: localhost:4317
|  Transport: gRPC
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.

Phonex UI http://localhost:6006/


In [49]:
# Init components
pdf_processor = PDFPRocessor(tracer)
vector_store = VectorStore(qdrant, embedder, tracer)
retriever = Retriver(embedder, qdrant, tracer)

co = cohere.Client(os.getenv("COHERE_API_KEY"))
reranker = Reranker(co, tracer)

genai.configure(api_key=os.getenv("api_key"))
model = genai.GenerativeModel("models/gemini-2.5-flash")

llm = LLM(model, tracer)
evaluator = Evaluator(llm, tracer)

rag = RAGPipeline(retriever, reranker, llm, evaluator)

# ---------------- UI ----------------

st.set_page_config(page_title="RAG Observability", layout="wide")
st.title("📊 RAG with Phoenix Observability")

# Upload PDF
uploaded_file = st.file_uploader("Upload PDF", type=["pdf"])

if uploaded_file:
    with open("temp.pdf", "wb") as f:
        f.write(uploaded_file.read())

    with st.spinner("Processing PDF..."):
        docs = pdf_processor.read_pdf("temp.pdf")
        chunks = pdf_processor.chunk_doc(docs)
        vector_store.store(chunks)

    st.success(f"Indexed {len(chunks)} chunks")

# Query
query = st.text_input("Ask a question")

if st.button("Run Query") and query:
    with st.spinner("Running RAG pipeline..."):
        result = rag.run(query)

    st.subheader("🧠 Answer")
    st.write(result["answer"])

    st.subheader("📊 Evaluation Score")
    st.write(result["score"])

# Debug
with st.expander("🔍 Debug"):
    st.write("Check Phoenix UI for full trace visualization")

2026-03-21 12:19:01.691 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-21 12:19:01.692 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-21 12:19:01.746 
  command:

    streamlit run /Users/asangtriratnaingle/Desktop/Codes_/embeddings/venv/lib/python3.12/site-packages/ipykernel_launcher.py [ARGUMENTS]
2026-03-21 12:19:01.746 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-21 12:19:01.746 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-21 12:19:01.747 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-21 12:19:01.747 Thread 'MainThread': missing ScriptRunCon